# BQIS - Production-Readiness Pipeline
### Dirancang supaya SIAP menerima data asli TÜV NORD, bukan cuma dummy dataset ini.

**4 hal yang dibenerin di sini dibanding notebook v1/v2 sebelumnya:**
1. **Column mapping** — nama kolom dari LIMS asli bisa beda, pipeline nggak boleh hardcode
2. **Threshold missing 30%** — sesuai proposal Layer 1 ("samples >30% missing dikecualikan & di-flag manual review"), TAPI ini belum pernah benar-benar diimplementasikan di v1/v2
3. **eps DBSCAN otomatis** (k-distance graph) — nggak boleh manual grid-search tiap ganti data
4. **Validasi fleksibel** — kalau ground truth kategori TIDAK tersedia (kemungkinan besar di data asli), tetap bisa jalan pakai Silhouette Score saja


In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

np.random.seed(42)

## 1. Column Mapping (Config, Bukan Hardcode)
Kalau data asli TÜV NORD nama kolomnya beda (misal dari LIMS mereka), cukup ubah dictionary
ini — SATU tempat, nggak perlu rewrite seluruh kode di bawahnya.

In [50]:
# Ganti value di kanan sesuai nama kolom asli dari sumber data baru.
# Key (kiri) = nama standar yang dipakai internal pipeline BQIS, JANGAN diubah.
COLUMN_MAP = {
    'sample_id':        'Sample_ID',
    'batch_code':       'Batch_Code',
    'product_name':     'Product_Name',
    'test_date':        'Test_Date',
    'moisture':         'Moisture_Content_%',
    'fat':              'Fat_Content_%',
    'protein':          'Protein_Content_%',
    'water_activity':   'Water_Activity_Aw',
    'acid_insol_ash':   'Acid_Insoluble_Ash_%',
    'acid_value':       'Acid_Value_mgKOHg',
    'peroxide':         'Peroxide_Value',
    'tpc':              'Total_Plate_Count_CFUg',
    'yeast_mold':       'Yeast_Mold_Count_CFUg',
    'lead':              'Lead_Pb_mgkg',
    'cadmium':          'Cadmium_Cd_mgkg',
    'status':           'Historical_Status',
    'failure_category': 'Failure_Category',   # opsional, boleh None kalau data asli nggak punya ini
}

NUMERIC_FEATURES = ['moisture', 'fat', 'protein', 'water_activity', 'acid_insol_ash',
                     'acid_value', 'peroxide', 'tpc', 'yeast_mold', 'lead', 'cadmium']

# Ambang batas resmi SNI 2973:2022 (ISI SESUAI ANGKA DI DOKUMEN SNI ASLI KALIAN).
# Ini placeholder -- isi manual dari dokumen SPC-TNI-020 / SNI 2973:2022 yang kalian punya.
SNI_LIMITS = {
    'moisture':       {'max': 5.0},   # contoh: {'max': 5.0}
    'acid_insol_ash': {'max': 0.1},
    'acid_value':     {'max': 1.0},
    'lead':           {'max': 0.5},
    'cadmium':        {'max': 0.2},
    'tpc':            {'max': 10000},
    'yeast_mold':     {'max': 1000},
}
print("Config siap. NUMERIC_FEATURES:", len(NUMERIC_FEATURES), "parameter")
print(">> TODO: isi SNI_LIMITS dengan angka baku mutu resmi sebelum dipakai ke data asli.")

Config siap. NUMERIC_FEATURES: 11 parameter
>> TODO: isi SNI_LIMITS dengan angka baku mutu resmi sebelum dipakai ke data asli.


## 2. Load & Standardize
Fungsi ini yang bakal dipanggil tiap kali ada dataset baru — tinggal ganti `path` dan
`COLUMN_MAP` di atas, logic di bawah nggak perlu disentuh.

In [51]:
def load_and_standardize(path, column_map=COLUMN_MAP):
    df_raw = pd.read_csv(path)
    reverse_map = {v: k for k, v in column_map.items() if v is not None}
    df_std = df_raw.rename(columns=reverse_map)

    missing_cols = [k for k in column_map if k not in df_std.columns and k != 'failure_category']
    if missing_cols:
        print(f"[WARNING] Kolom berikut tidak ditemukan di data baru: {missing_cols}")
        print("          Cek COLUMN_MAP -- kemungkinan nama kolom sumber berbeda.")
    return df_std

df = load_and_standardize("../data/bqis_biscuit_quality_dataset.csv")
print(df.shape)
df.head(3)

(1000, 17)


,sample_id,batch_code,product_name,test_date,moisture,fat,protein,water_activity,acid_insol_ash,acid_value,peroxide,tpc,yeast_mold,lead,cadmium,failure_category,status
0,SPL-2026-0409,B-250101-4,Wafer Vanilla,2025-01-01,3.23,19.10,9.95,0.543,0.058,1.61,1.34,5670,2232,0.045,0.098,NaN,Pass
1,SPL-2026-0949,B-250101-5,Wafer Vanilla,2025-01-01,1.23,17.81,4.76,0.375,0.045,1.13,1.05,5847,272,0.014,0.018,NaN,Pass
2,SPL-2026-0482,B-250101-7,Cookies Choco Chip,2025-01-01,3.69,16.93,NaN,0.620,0.058,0.68,1.07,27579,162,0.363,0.162,NaN,Pass


## 3. Missing-Value Handling Sesuai Proposal (Threshold 30%)
Ini implementasi konkret dari klaim di Layer 1 proposal: *"Samples with more than 30%
missing parameter values are excluded from AI analysis and flagged for manual review."*
Sebelumnya (v1/v2) klaim ini ADA di dokumen tapi BELUM pernah benar-benar dijalankan di kode.

In [52]:
def flag_and_impute(df, numeric_features=NUMERIC_FEATURES, missing_threshold=0.30, n_neighbors=5):
    missing_pct = df[numeric_features].isna().sum(axis=1) / len(numeric_features)
    df = df.copy()
    df['flag_manual_review'] = missing_pct > missing_threshold

    n_flagged = df['flag_manual_review'].sum()
    print(f"Sampel di-flag manual review (missing >{missing_threshold*100:.0f}%): {n_flagged}/{len(df)} ({n_flagged/len(df)*100:.1f}%)")

    df_clean = df[~df['flag_manual_review']].reset_index(drop=True)
    df_flagged = df[df['flag_manual_review']].reset_index(drop=True)

    imputer = KNNImputer(n_neighbors=n_neighbors, weights='distance')
    df_clean[numeric_features] = imputer.fit_transform(df_clean[numeric_features])

    return df_clean, df_flagged

df_clean, df_flagged = flag_and_impute(df)
print(f"\nSiap dianalisis AI: {len(df_clean)} sampel")
print(f"Perlu manual review auditor: {len(df_flagged)} sampel (TIDAK diproses AI, sesuai proposal)")

Sampel di-flag manual review (missing >30%): 0/1000 (0.0%)

Siap dianalisis AI: 1000 sampel
Perlu manual review auditor: 0 sampel (TIDAK diproses AI, sesuai proposal)


## 4. Cross-Check ke Baku Mutu SNI (Bukan Cuma Statistik)
Ini yang belum ada sama sekali di v1/v2 — pipeline sekarang cuma belajar pola statistik,
padahal kalian sudah punya dokumen SNI 2973:2022 dengan angka baku mutu resmi.
Menjangkarkan hasil ke standar resmi = argumen kuat untuk kredibilitas ke auditor TÜV NORD.

In [53]:
def check_sni_compliance(df, sni_limits=SNI_LIMITS):
    results = {}
    for param, limit in sni_limits.items():
        if limit.get('max') is None or param not in df.columns:
            continue
        exceed = (df[param] > limit['max']).sum()
        results[param] = {'n_exceed': exceed, 'pct_exceed': round(exceed/len(df)*100, 1)}
    if not results:
        print("[INFO] SNI_LIMITS masih kosong (placeholder). Isi dulu angka baku mutu resminya di Section 1.")
    else:
        for param, r in results.items():
            print(f"{param}: {r['n_exceed']} sampel melebihi batas SNI ({r['pct_exceed']}%)")
    return results

sni_check = check_sni_compliance(df_clean)

moisture: 61 sampel melebihi batas SNI (6.1%)
acid_insol_ash: 104 sampel melebihi batas SNI (10.4%)
acid_value: 467 sampel melebihi batas SNI (46.7%)
lead: 30 sampel melebihi batas SNI (3.0%)
cadmium: 37 sampel melebihi batas SNI (3.7%)
tpc: 169 sampel melebihi batas SNI (16.9%)
yeast_mold: 342 sampel melebihi batas SNI (34.2%)


## 5. Feature Selection — Otomatis, Bukan Hardcode "Top-5"
Kalau `failure_category` TERSEDIA (kayak dataset dummy ini), pakai Mutual Information.
Kalau TIDAK TERSEDIA (kemungkinan besar di data asli), fallback ke variance-based selection
(fitur dengan variansi tertinggi setelah scaling -- proxy sederhana untuk "fitur paling
membedakan sampel").

In [54]:
def select_features(df, numeric_features=NUMERIC_FEATURES, status_col='status',
                     category_col='failure_category', top_n=5):
    fail_mask = df[status_col].map({'Pass': 0, 'Fail': 1}) == 1
    X_fail = df.loc[fail_mask, numeric_features].reset_index(drop=True)

    has_ground_truth = category_col in df.columns and df[category_col].notna().any()

    if has_ground_truth:
        cat_ref = df.loc[fail_mask, category_col].reset_index(drop=True)
        mi_scores = mutual_info_classif(X_fail, cat_ref, random_state=42)
        ranking = pd.Series(mi_scores, index=numeric_features).sort_values(ascending=False)
        method = "Mutual Information (ground truth tersedia)"
    else:
        scaler_tmp = StandardScaler()
        X_scaled_tmp = scaler_tmp.fit_transform(X_fail)
        ranking = pd.Series(X_scaled_tmp.var(axis=0), index=numeric_features).sort_values(ascending=False)
        cat_ref = None
        method = "Variance-based (FALLBACK -- ground truth TIDAK tersedia)"

    print(f"Metode feature selection: {method}")
    print(ranking.round(4))

    selected = ranking.head(top_n).index.tolist()
    return X_fail, selected, cat_ref, has_ground_truth

X_fail, selected_features, cat_ref, has_gt = select_features(df_clean)
print("\nFitur terpilih:", selected_features)

Metode feature selection: Mutual Information (ground truth tersedia)
acid_value        0.3183
cadmium           0.1965
lead              0.1481
yeast_mold        0.1177
peroxide          0.0797
protein           0.0646
acid_insol_ash    0.0518
moisture          0.0282
tpc               0.0010
fat               0.0000
water_activity    0.0000
dtype: float64

Fitur terpilih: ['acid_value', 'cadmium', 'lead', 'yeast_mold', 'peroxide']


## 6. eps DBSCAN Otomatis (k-distance Graph)
Ganti grid-search manual (yang kemarin butuh trial-error tiap ganti dataset) dengan metode
data-driven: plot jarak ke tetangga ke-k, cari titik "siku" (elbow) -- itu jadi kandidat eps.

In [55]:
def suggest_eps(X_scaled, k=5, candidate_range=None):
    from sklearn.cluster import DBSCAN as _DBSCAN
    
    if candidate_range is None:
        neighbors = NearestNeighbors(n_neighbors=k)
        neighbors.fit(X_scaled)
        distances, _ = neighbors.kneighbors(X_scaled)
        k_distances = np.sort(distances[:, -1])
        # ambil rentang dari persentil 10-90% jarak k-NN sebagai kandidat wajar
        lo, hi = np.percentile(k_distances, [10, 90])
        candidate_range = np.linspace(lo, hi, 15)

    best_eps, best_score, results = None, -1, []
    for eps in candidate_range:
        labels = _DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct = (labels == -1).sum() / len(labels)

        # skip kandidat degenerate: cuma 1 cluster, atau noise ekstrem
        if n_clusters < 2 or noise_pct > 0.5 or noise_pct < 0.05:
            results.append((eps, n_clusters, noise_pct, None))
            continue

        score = silhouette_score(X_scaled, labels)
        results.append((eps, n_clusters, noise_pct, score))
        if score > best_score:
            best_score, best_eps = score, eps

    print("eps      | n_clusters | noise%  | silhouette")
    for eps, nc, np_, sc in results:
        sc_str = f"{sc:.3f}" if sc is not None else "  (skip)"
        print(f"{eps:.3f}  | {nc:^10} | {np_*100:5.1f}% | {sc_str}")

    if best_eps is None:
        print("\n[WARNING] Tidak ada kandidat eps yang wajar (semua degenerate).")
        print("          Rekomendasi: gunakan K-Means saja untuk dataset ini,")
        print("          DBSCAN tidak cocok untuk struktur density data ini.")
        return None

    print(f"\nEps terpilih (silhouette tertinggi, kandidat valid): {best_eps:.3f}")
    return best_eps

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_fail[selected_features])
eps_suggested = suggest_eps(X_scaled, k=5)

eps      | n_clusters | noise%  | silhouette
0.503  |     4      |  82.8% |   (skip)
0.569  |     3      |  71.0% |   (skip)
0.634  |     1      |  62.9% |   (skip)
0.700  |     3      |  51.7% |   (skip)
0.766  |     4      |  40.2% | -0.025
0.831  |     1      |  29.6% |   (skip)
0.897  |     1      |  26.4% |   (skip)
0.962  |     1      |  21.6% |   (skip)
1.028  |     1      |  19.3% |   (skip)
1.093  |     1      |  16.1% |   (skip)
1.159  |     2      |  13.2% | 0.180
1.224  |     2      |  10.1% | 0.248
1.290  |     1      |   6.3% |   (skip)
1.355  |     1      |   4.9% |   (skip)
1.421  |     1      |   4.6% |   (skip)

Eps terpilih (silhouette tertinggi, kandidat valid): 1.224


## 7. Clustering + Validasi Fleksibel
Kalau ground truth ADA -> laporkan ARI & NMI (seperti v1/v2).
Kalau ground truth TIDAK ADA -> laporkan Silhouette Score saja + serahkan ke auditor untuk
validasi kualitatif per cluster (bukan angka, tapi tetap actionable).

In [56]:
def cluster_and_validate(X_scaled, eps, cat_ref=None, has_ground_truth=False, k_range=range(2,7)):
    # K-Means: pilih k via silhouette (selalu bisa dihitung, tidak butuh ground truth)
    sil_scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        sil_scores[k] = silhouette_score(X_scaled, labels)
    best_k = max(sil_scores, key=sil_scores.get)

    kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    kmeans_labels = kmeans_final.fit_predict(X_scaled)

    db_final = DBSCAN(eps=eps, min_samples=5)
    dbscan_labels = db_final.fit_predict(X_scaled)
    n_noise = list(dbscan_labels).count(-1)

    print(f"K-Means: k={best_k} (silhouette={sil_scores[best_k]:.3f})")
    print(f"DBSCAN: eps={eps:.3f}, noise={n_noise}/{len(X_scaled)} ({n_noise/len(X_scaled)*100:.1f}%)")

    if has_ground_truth and cat_ref is not None:
        print(f"\n[Ground truth tersedia] Validasi kuantitatif:")
        print(f"  ARI K-Means: {adjusted_rand_score(cat_ref, kmeans_labels):.3f}")
        print(f"  NMI K-Means: {normalized_mutual_info_score(cat_ref, kmeans_labels):.3f}")
        print(f"  ARI DBSCAN : {adjusted_rand_score(cat_ref, dbscan_labels):.3f}")
        print(f"  NMI DBSCAN : {normalized_mutual_info_score(cat_ref, dbscan_labels):.3f}")
    else:
        print(f"\n[Ground truth TIDAK tersedia] Validasi kualitatif diperlukan:")
        print("  -> Silhouette Score di atas jadi satu-satunya sinyal kuantitatif.")
        print("  -> Sampel per cluster perlu direview auditor pangan untuk sanity-check domain.")

    return kmeans_labels, dbscan_labels, best_k

kmeans_labels, dbscan_labels, best_k = cluster_and_validate(X_scaled, eps_suggested, cat_ref, has_gt)

K-Means: k=5 (silhouette=0.321)
DBSCAN: eps=1.224, noise=35/348 (10.1%)

[Ground truth tersedia] Validasi kuantitatif:
  ARI K-Means: 0.398
  NMI K-Means: 0.448
  ARI DBSCAN : 0.215
  NMI DBSCAN : 0.149


## Ringkasan
Pipeline ini dirancang supaya kalau dataset dummy diganti data asli TÜV NORD:
- **Ganti `COLUMN_MAP`** kalau nama kolom sumber beda
- **Isi `SNI_LIMITS`** dengan angka baku mutu resmi dari SNI 2973:2022
- Sisanya (missing threshold, feature selection, eps, clustering, validasi) **otomatis
  menyesuaikan** tanpa perlu rewrite kode atau trial-error manual seperti di v1/v2.
